# extract

> what a document *is*, the fields inside it, and an answer over the whole of it

In [ ]:
#| default_exp extract

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Retrieval answers where something is. This module answers what a vault should *do* with what
[varga](https://vedicreader.github.io/varga/) worked out: the doctype written into a document's
meta, the shelf it implies, and an extraction run against the vault's own model.


In [ ]:
#| export
from collections import Counter
from fastcore.all import AttrDict, L, patch
from rishi.core import infer_runtime
from rahasya import NER_CHARS, pii_report, redact, redact_obj
from varga import (DOCTYPES, KIND_BONUS, KIND_HINT, MIN_MARGIN, MIN_SCORE, SIGNALS, TYPE_SP,
                   cue_scores, guess_type, signals)
from varga.schema import (EXTRACT_SP, FIELD_TYPES, SCHEMAS, Catalogue, Contract, Invoice,
                          MeetingNotes, Paper, Receipt, Resume, Summary, as_schema, dyn_schema,
                          schema_str, structured)
from vishalakshi.core import Vault
from vishalakshi.ask import (LOCAL_RUNTIMES, PII_SP, dflt_model, is_stock_chat,
                            new_chat, pii_model_)   # also patches Vault.ask

# re-exported: what varga decides is still reachable from the vault that stores the verdict
_all_ = ['DOCTYPES', 'EXTRACT_SP', 'FIELD_TYPES', 'KIND_BONUS', 'KIND_HINT', 'MIN_MARGIN',
         'MIN_SCORE', 'NER_CHARS', 'SCHEMAS', 'SIGNALS',
         'TYPE_SP', 'Catalogue', 'Contract', 'Invoice', 'MeetingNotes', 'Paper', 'Receipt',
         'Resume', 'Summary', 'as_schema', 'cue_scores', 'dyn_schema', 'guess_type', 'schema_str',
         'signals', 'structured']


`categorize` writes the verdict into the document's `meta`. `score` and `decisive` are the seam: a clear winner needs no model; a tie is what `llm='auto'` spends one on. `by` records which leg answered. `force=False` only looks at what arrived since the last run.


In [ ]:
#| export
def model_cached(mid:str) -> bool:
    'Is this model already in the Hugging Face cache? A cache scan, never a download.'
    try:
        from huggingface_hub import scan_cache_dir
        return any(r.repo_id == mid for r in scan_cache_dir().repos)
    except Exception: return False

@patch
def categorize(self:Vault,
               ref,                # doc_id, source, title, a path on disk, or a loaded `document()`
               model:str=None,     # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
               chat_kw:dict=None,  # anything else rishi's `Chat` takes: temp, runtime, think, …
               llm:str='auto',     # 'auto' -> only when the cues cannot decide | 'always' | 'never'
               labels:str=None,    # comma-separated labels to choose from; None -> DOCTYPES
               max_chars:int=6000, # chars of the document the model and the cues see
               ner:bool=True,      # run entity extraction alongside the regex signals
               save:bool=True,     # write the verdict into the document's meta
) -> AttrDict:
    'What kind of document this is: invoice, catalogue, contract, paper, transcript, code…'
    d = (ref if isinstance(ref, dict) and 'text' in ref
         else self.document(ref, max_chars=max(max_chars, 4000)))
    txt = (d.text or '')[:max_chars]
    if not txt.strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, doctype=None, skipped='no text to judge')
    sig = signals(txt, ner=ner)
    g = guess_type(txt, sig, kind=d.kind)
    dt, by = g.doctype, f'cues ({g.method})'
    mode = {True: 'always', False: 'never', None: 'auto'}.get(llm, str(llm).lower())
    assert mode in ('auto', 'always', 'never'), f"llm must be auto, always or never, not {llm!r}"
    if mode == 'always' or (mode == 'auto' and not g.decisive):
        lbls = L(labels.split(',') if isinstance(labels, str) else labels or list(DOCTYPES)).map(str.strip)
        mid = model or dflt_model
        try:
            # auto may use a model already loaded; it must not download one
            if mode == 'auto' and is_stock_chat() and infer_runtime(mid) != 'remote' and not model_cached(mid):
                by = f'cues ({g.method}); {mid or "no model"} is not downloaded, so none was asked'
            else:
                said = new_chat(mid, **(chat_kw or {})).classify(f'{d.title}\n\n{txt}', list(lbls)+['other'], sp=TYPE_SP)
                if said in lbls or said == 'other': dt, by = said, f'llm ({mid or "rishi default"})'
                else: by = f'cues ({g.method}); llm answered {said[:40]!r}, not a label'
        except Exception as e:
            # keep the cue verdict when the model leg fails
            if mode == 'always': raise
            by = f'cues ({g.method}); no model available ({type(e).__name__})'
    res = AttrDict(doc_id=d.doc_id, title=d.title, kind=d.kind, doctype=dt, score=g.score,
                   margin=g.margin, decisive=g.decisive, by=by, scores=g.scores,
                   signals=sig.counts, ents=sig.ents, saved=False)
    if save and d.doc_id:
        self.set_meta(d.doc_id, doctype=dt, doctype_by=by, doctype_score=g.score)
        res.saved = True
    return res

@patch
def categorize_all(self:Vault,
                   kind:str=None,     # restrict to one or more KINDS
                   force:bool=False,  # re-type documents that already carry a doctype
                   limit:int=None,    # stop after this many
                   **kw               # forwarded to categorize (model=, chat_kw=, llm=, ner=, max_chars=)
) -> AttrDict:
    'Type every document in the vault that is not typed yet, and report the shape of the corpus.'
    docs = self.sources(kind)
    if not force: docs = docs.filter(lambda r: not (r['meta'] or {}).get('doctype'))
    out = L()
    for r in docs[:limit]:
        try: out.append(self.categorize(r['id'], **kw))
        except Exception as e:
            out.append(AttrDict(doc_id=r['id'], title=r['title'], doctype=None,
                                error=f'{type(e).__name__}: {str(e)[:200]}'))
    return AttrDict(n=len(out), by_type=dict(Counter(o.doctype for o in out).most_common()),
                    errors=out.filter(lambda o: o.get('error')), results=out)

@patch
def doctypes(self:Vault, kind:str=None) -> dict:
    'How many documents of each type the vault holds. `untyped` counts the ones never categorised.'
    c = Counter((r['meta'] or {}).get('doctype') or 'untyped' for r in self.sources(kind))
    return dict(c.most_common())

@patch
def of_type(self:Vault, doctype:str, kind:str=None) -> L:
    'Every document categorised as `doctype`, newest first.'
    return self.sources(kind).filter(lambda r: (r['meta'] or {}).get('doctype') == doctype)

# doctype routing (KIND_SHELF is arrival kind)
DOCTYPE_SHELF = {'paper': 'papers', 'code': 'code', 'catalogue': 'data'}

@patch
def reshelf(self:Vault,
            ref,                      # doc_id, source, or a title substring
            shelf:str=None,           # where to put it; None -> whatever its doctype says
            llm:str='auto',           # the LLM leg of the categorisation that picks the shelf
            max_chars:int=2_000_000,  # cap on the text carried over
            **kw                      # forwarded to categorize (model=, chat_kw=, ner=)
) -> AttrDict:
    'Move a document to the shelf its *type* says it belongs on: the route that had to wait.'
    d = self.document(ref, max_chars=max_chars)
    c = self.categorize(d, save=False, llm=llm, **kw) if shelf is None else None
    nm = shelf or DOCTYPE_SHELF.get(c.doctype)
    out = AttrDict(doc_id=d.doc_id, title=d.title, doctype=c.doctype if c else None,
                   was=self.name, store=self.name, moved=False)
    if not nm or nm == self.name or not d.doc_id: return out
    r = self.shelf(nm).add(d.text, d.title, source=d.source, kind=d.kind, force=True,
                           meta=dict(d.meta, doctype=out.doctype, reshelved_from=self.name))
    self.forget(d.doc_id)
    return AttrDict(out, doc_id=r.get('doc_id'), store=nm, moved=True, chunks=r.get('chunks'))

@patch
def ner(self:Vault,
        ref:str,              # doc_id, source, title substring, or a path on disk
        limit:int=40,         # entities returned
        max_chars:int=20000,
) -> AttrDict:
    'What one document names: organisations, people, places, products, and the terms around them.'
    d = self.document(ref, max_chars=max_chars)
    sig = signals(d.text, limit=limit)
    return AttrDict(doc_id=d.doc_id, title=d.title, method=sig.method, counts=sig.counts,
                    labels=sig.labels, ents=sig.ents)


## Extraction

`extract` reads a whole document and returns a dict. No `schema` means categorise first and take the doctype's shape. rishi constrains the model (tool call / grammar / parsed JSON by backend). `extract_all(doctype='invoice')` returns one row per match.


In [ ]:
#| export
@patch
def extract(self:Vault,
            ref:str,               # doc_id, source, title substring, or a path on disk
            schema=None,           # a SCHEMAS key, a dataclass, or a 'field:type, …' spec; None -> from the doctype
            model:str=None,        # an id, a path, `mlx/…`; None -> $VISHALAKSHI_MODEL
            chat_kw:dict=None,     # anything else rishi's `Chat` takes: temp, runtime, think, …
            max_chars:int=8000,    # chars of the document the model sees
            sp:str=EXTRACT_SP,     # system prompt for the extraction
            save:bool=False,       # write the fields into the document's meta
            llm:str='auto',        # the LLM leg of the categorisation, when picking the schema
            pii:str='local',       # local|redact|refuse|off; same contract as `ask`
            pii_model:str=None,    # local model when private; None -> $VISHALAKSHI_PII_MODEL
) -> AttrDict:
    "Pull structured fields out of one document: an invoice's totals, a catalogue's products."
    d = self.document(ref, max_chars=max_chars)
    if not (d.text or '').strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=None, schema=None,
                        fields={}, skipped='no text to extract from')
    # marks win over arithmetic, so this is `self.pii` rather than a bare `pii_report`
    report = (self.pii(d.doc_id, max_chars=max_chars) if d.doc_id else pii_report(d.text)) if pii != 'off' else None
    private, text = bool(report and report.has_pii), d.text
    if private and pii == 'refuse':
        return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=None, schema=None,
                        fields={}, skipped='pii', refused=True, pii=report)
    if private and pii == 'redact': text = redact(d.text)
    dt, cat = None, None
    if schema is None:
        cat = self.categorize(d, model=model, chat_kw=chat_kw, llm=llm, save=save)
        dt = cat.doctype
        sch = SCHEMAS.get(dt, Summary)
    else: sch = as_schema(schema)
    prompt = (f'{d.title}\n(source: {d.source})\n\n{text}\n\n---\n\n'
              f'Pull the fields of `{sch.__name__}` out of the document above.')
    mid, sys_sp = model, sp
    if private and pii == 'local':
        mid, sys_sp = pii_model or pii_model_, PII_SP
    ch = new_chat(mid, **(chat_kw or {}))
    if private and pii == 'local' and str(getattr(ch, 'runtime', '') or '') not in LOCAL_RUNTIMES:
        return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=dt, schema=None,
                        fields={}, skipped='pii', refused=True, pii=report, runtime=ch.runtime)
    flds = structured(ch, prompt, sch, sp=sys_sp)
    # the same backstop `ask` applies to an answer: fields are short, so names are looked for
    if private and pii_report(str(flds), ner=True).has_pii: flds = redact_obj(flds, ner=True)
    if save and d.doc_id: self.set_meta(d.doc_id, extracted=flds, extracted_as=sch.__name__)
    return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=dt, schema=sch.__name__,
                    fields=flds, chars=len(d.text), truncated=d.truncated, model=mid or dflt_model,
                    runtime=ch.runtime, categorized=cat, usage=getattr(ch, 'use', None), pii=report)

@patch
def extract_all(self:Vault,
                doctype:str=None,   # only documents already categorised as this
                kind:str=None,      # only these KINDS
                schema=None,        # one shape for all of them; None -> each document's own doctype
                limit:int=None,     # stop after this many
                **kw                # forwarded to extract (model=, chat_kw=, max_chars=, save=, sp=)
) -> AttrDict:
    'Extract from many documents at once: a folder of invoices as one table.'
    docs = self.of_type(doctype, kind) if doctype else self.sources(kind)
    rows, errs = L(), L()
    for r in docs[:limit]:
        try:
            e = self.extract(r['id'], schema=schema, **kw)
            if e.get('skipped'): errs.append(dict(doc_id=r['id'], title=r['title'], error=e.skipped))
            else: rows.append(dict({'doc_id': e.doc_id, 'doc_title': e.title, 'schema': e.schema},
                                   **e.fields))
        except Exception as ex:
            errs.append(dict(doc_id=r['id'], title=r['title'], error=f'{type(ex).__name__}: {str(ex)[:200]}'))
    return AttrDict(n=len(rows), doctype=doctype, schema=schema, rows=rows, errors=errs)


## Asking over one whole document

`ask_doc` / `ref=` pin named documents and retrieve a few neighbours behind them. `schema=` turns the answer into a structured response. `doc_chars` is the budget for the named documents together.


In [ ]:
#| export
@patch
def ask_doc(self:Vault,
            ref:str,              # doc_id, source, title substring, or a path on disk
            question:str,         # what you want to know about it
            schema=None,          # answer as this shape instead of prose
            max_chars:int=8000,   # chars of the document the model sees
            related:int=3,        # sections from the *rest* of the vault to add as context
            **kw                  # forwarded to `ask` (model=, chat_kw=, sp=)
) -> AttrDict:
    'Answer a question about one whole document: `ask` with that document as section [1].'
    return self.ask(question, ref=ref, schema=schema, doc_chars=max_chars, related=related, **kw)

## Try it

Two documents, one of each kind, and nothing here needs a model yet: the cue table, the signals and
the schema machinery are the legs that run everywhere.

In [ ]:
#| eval:false
from vishalakshi import Vault

INVOICE = '''# INVOICE

Invoice No: ACM-2024-0117
Date: 2024-03-01
Payment terms: Net 30

Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd

## Line items

| Description | Qty | Unit price | Amount |
|---|---|---|---|
| Widget, steel | 12 | $8.50 | $102.00 |
| Gasket, nitrile | 40 | $1.20 | $48.00 |

Subtotal: $150.00
VAT (20%): $30.00
Total due: $180.00
'''

CATALOGUE = '''# Spring price list

## Widget, steel

SKU: WS-100. Price: $8.50 per unit. In stock.
Specifications: 40mm, zinc plated. Dimensions 40x12x4mm.

## Gasket, nitrile

SKU: GN-220. Price: $1.20 per unit. Out of stock.
Add to cart to be notified.
'''

v = Vault(':memory:')
v.add(INVOICE, 'Acme invoice ACM-2024-0117', source='/inbox/acme-0117.md')
v.add(CATALOGUE, 'Spring price list', source='/inbox/spring-list.md')
v.doctypes()

{'untyped': 2}

In [ ]:
#| eval: false
r = v.categorize('Acme invoice ACM-2024-0117', llm='never')
r.doctype, r.score, r.decisive, r.by

('invoice', 1.0, True, 'cues (ner+regex)')

In [ ]:
#| eval: false
test_eq(r.doctype, 'invoice')
assert r.decisive, r.scores          # the cues are sure enough that no model is needed
test_eq(v.doc(r.doc_id)['meta']['doctype'], 'invoice')   # written back into the vault
test_eq(v.categorize('Spring price list', llm='never').doctype, 'catalogue')
test_eq(v.doctypes(), {'invoice': 1, 'catalogue': 1})
test_eq(v.of_type('invoice').attrgot('title'), ['Acme invoice ACM-2024-0117'])

In [ ]:
#| eval:false
#| hide
# the regex leg owns the numeric evidence and runs regardless of whether entities are found
sig = signals(INVOICE)
assert sig.method in ('ner+regex', 'keyphrase+regex', 'regex'), sig.method
for k in ('money', 'date', 'ref', 'tax', 'percent', 'table', 'heading'): assert sig.counts.get(k), k
test_eq(signals(INVOICE, ner=False).method, 'regex')
test_eq(signals(INVOICE, ner=False).ents, [])

# the handrolled entity leg finds company names by legal suffix
if sig.method == 'ner+regex':
    assert any('Acme' in e.text or 'Contoso' in e.text for e in sig.ents), sig.ents
    assert sig.labels.get('org'), sig.labels
    # labelled entities sort before keyphrases
    assert all(e.label not in ('KEYPHRASE', 'TERM') for e in sig.ents[:len(sig.ents.filter(lambda e: e.label not in ('KEYPHRASE','TERM')))])

# `limit` caps the list, never the evidence: a score must not move with it
test_eq(cue_scores(INVOICE, signals(INVOICE, limit=2))['invoice'], cue_scores(INVOICE, sig)['invoice'])
test_eq(len(signals(INVOICE, limit=2).ents), 2)

# a score is a fraction, never a count: the same evidence twice over must not score higher
test_eq(cue_scores(INVOICE)['invoice'], cue_scores(INVOICE + INVOICE)['invoice'])
# and what acquired a document is evidence about what it is
test_eq(guess_type('Um, so, welcome back. 00:12 speaker 1: right.', kind='youtube').doctype, 'transcript')

# a document with nothing to go on says so rather than picking a type at random
g = guess_type('The cat sat on the mat.')
assert not g.decisive and g.score < MIN_SCORE, g
test_eq(v.categorize(AttrDict(doc_id=None, title='t', kind=None, text='   '),
                     llm='never').skipped, 'no text to judge')
# three states, and a word for each: a `bool` cannot carry the third past argparse
test_fail(lambda: v.categorize('Acme invoice', llm='sometimes'), contains='auto, always or never')

The cells above run without a hosted key. Where cues cannot decide, or nested `items` need a bigger model, pass `model=` / `chat_kw=` the same way as `ask`. Measured fallback behaviour is in `evals/`.


In [ ]:
#| hide
from functools import partial
from fastcore.all import Path
from vishalakshi.ask import CachedChat, use_chat
from rishi.litert import gemma4_e4b

CACHE = (Path('nbs') if Path('nbs').exists() else Path('.'))/'chatcache'
_replay = use_chat(partial(CachedChat, path=CACHE)); _replay.__enter__()

INV = '''# INVOICE

Invoice No: ACM-2024-0117
Date: 2024-03-01
Payment terms: Net 30

Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd

## Line items

| Description | Qty | Unit price | Amount |
|---|---|---|---|
| Widget, steel | 12 | $8.50 | $102.00 |
| Gasket, nitrile | 40 | $1.20 | $48.00 |

Subtotal: $150.00
VAT (20%): $30.00
Total due: $180.00
'''
# offline: the hashing encoder is deterministic, so a retrieved prompt is byte-identical on replay
_x = Vault(':memory:', offline=True)
_x.add(INV, 'Acme invoice ACM-2024-0117', source='/inbox/acme-0117.md')
_x.add('   ', 'empty one', source='/inbox/empty.md')
# just the id: `new_chat` asks LiteRT for the GPU on its own, and `eng_kw=dict(backend=…)`
# collides with the `backend=be` that `create_engine` already passes `Engine`
_m = dict(model=gemma4_e4b)

In [ ]:
#| hide
from rishi.core import Chat
# the LLM leg of categorisation, against what gemma-4-E2B actually replied
c = _x.categorize('/inbox/acme-0117.md', llm='always', save=False, **_m)
assert c.by.startswith('llm'), c.by
assert 'Total due' in _x.document('/inbox/acme-0117.md').text     # the model saw the whole document

# `auto` may use a model, never fetch one: an uncached id is declined before a chat is built
test_eq(model_cached('no-such-org/no-such-model'), False)
_und = AttrDict(doc_id=None, title='untitled', kind=None, text='The cat sat on the mat.')
with use_chat(Chat):   # the guard asks whether the *stock* backend would have to fetch weights
    _by = _x.categorize(_und, model='no-such-org/no-such-model', save=False).by
assert 'is not downloaded' in _by, _by
# ...and the cue verdict is what stands when no model answers, with `by` saying so
assert _by.startswith('cues ('), _by


In [ ]:
#| hide
import json, warnings
from dataclasses import fields
# extraction with no schema: the doctype picks `Invoice`, and the nested `items` is the field
# a small model is least reliable on
e = _x.extract('/inbox/acme-0117.md', **_m)
test_eq((e.schema, e.doctype), ('Invoice', 'invoice'))
test_eq(e.fields['total'], 180.0)                       # what the model read off the document
test_eq(e.fields['number'], 'ACM-2024-0117')            # E2B returns '' here; E4B reads it
test_eq(len(e.fields['items']), 2)                      # the nested list, one entry per line item
test_eq(e.runtime, 'litert')

# LiteRT used to refuse to parse the tool call gemma emitted for a nested `items`, and that
# refusal *was* this test; the installed runtime no longer refuses, on either E2B or E4B. The
class _Refuses:
    "Constrained call fails; the unconstrained retry answers in a JSON fence, as a real one does."
    runtime, use, hist = 'litert', None, []
    def structured(self, prompt, schema, sp=''):
        raise RuntimeError('litert_lm_conversation failed to parse the tool call')
    def __call__(self, prompt, **kw):
        return dict(role='assistant', content=[dict(type='text', text=
            '```json\n{"number": "ACM-2024-0117", "total": 180.0, "items": '
            '[{"description": "Widget, steel"}, {"description": "Gasket, nitrile"}]}\n```')])
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    _f = structured(_Refuses(), 'the invoice', Invoice, sp=EXTRACT_SP)
assert any('retrying as a JSON reply' in str(x.message) for x in w), [str(x.message) for x in w]
test_eq((_f['number'], _f['total'], len(_f['items'])), ('ACM-2024-0117', 180.0, 2))

# a schema named in one string, and the flat fields a 2B model is reliable on
f = _x.extract('/inbox/acme-0117.md', schema='vendor:str, total:float', **_m).fields
test_eq(sorted(f), ['total', 'vendor'])
test_eq(f['total'], 180.0)
# a document with nothing in it is skipped before any model is asked
test_eq(_x.extract('/inbox/empty.md', **_m).skipped, 'no text to extract from')
# across the corpus, one row per document, with identity columns that cannot shadow a schema's
# own
test_eq(_x.categorize('/inbox/acme-0117.md', llm='never').doctype, 'invoice')
b = _x.extract_all(doctype='invoice', schema='vendor:str, total:float', **_m)
test_eq((b.n, len(b.errors)), (1, 0))
test_eq(sorted(b.rows[0]), ['doc_id', 'doc_title', 'schema', 'total', 'vendor'])


In [ ]:
#| hide
# ask_doc: the document is section [1], so `ask`'s citation contract holds: [n] resolves to
# something read() can open
a = _x.ask_doc('/inbox/acme-0117.md', 'what is the total due?', related=2, **_m)
assert a.answer, a
test_eq(a.cited.attrgot('node_id')[0], f'{a.doc_id}#0')
assert 'VAT (20%)' in a.prompt, 'the whole document must reach the model, not a retrieved chunk'
assert a.prompt.endswith('Question: what is the total due?')
# the same question as data instead of prose, through a schema built at the moment of asking
d = _x.ask_doc('/inbox/acme-0117.md', 'what is owed and to whom?',
               schema='amount:float, currency:str, owed_to:str', **_m)
test_eq((d.schema, sorted(d.fields), d.answer), ('Answer', ['amount', 'currency', 'owed_to'], None))
test_eq(d.fields['amount'], 180.0)
# and it is `ask` underneath
test_eq(_x.ask('what is the total due?', ref='/inbox/acme-0117.md', related=2, **_m).answer, a.answer)

# a file the vault has never seen, straight off disk: no model needed to prove where it came from
from fastcore.all import Path
from tempfile import mkdtemp
_p = Path(mkdtemp())/'notes.md'
_p.write_text('# Standup\n\nAttendees: Ana, Bo. Action items: Ana to ship the parser.')
test_eq(_x.document(_p).origin, 'disk')
test_eq(_x.categorize(str(_p), llm='never', save=False).doctype, 'meeting_notes')

/Users/71293/code/personal/orgs/vishalakshi/vishalakshi/extract.py:584: UserWarning: RuntimeError on a constrained call for Answer (ValueError: model neither called the tool nor returned JSON; reply: 'The total amount owed is $180.00 [1]. This amount is owed to Acme Supplies Ltd [1) — retrying as a JSON reply.
  warnings.warn(f'{type(e).__name__} on a constrained call for {schema.__name__} '


In [ ]:
#| eval:false
#| hide
# the route that had to wait: what acquisition could not tell at the door, the doctype can
v.shelf('papers', offline=True)          # pre-registered, so this test pays for no encoder download
r = v.reshelf('Acme invoice', llm='never')
test_eq((r.doctype, r.moved), ('invoice', False))   # an invoice has no shelf of its own: it stays put

PAPER = ('# Late chunking\n\n## Abstract\n\nWe propose contextual chunk embeddings.\n\n'
         '## Introduction\n\nRelated work [1] et al. improved on this.\n\n## References\n\ndoi:10.1/x')
v.add(PAPER, 'late chunking', source='/in/lc.pdf')
r = v.reshelf('/in/lc.pdf', llm='never')
test_eq((r.doctype, r.was, r.store, r.moved), ('paper', 'store', 'papers', True))
test_eq(v.doc('/in/lc.pdf'), None)                                    # gone from the shelf it was on
assert v.shelf('papers').document('/in/lc.pdf').text.startswith('# Late chunking')
test_eq(v.shelf('papers').doc('/in/lc.pdf')['meta']['doctype'], 'paper')   # and it remembers why
test_eq(v.reshelf('Spring price list', shelf='papers').moved, True)        # named: no categorisation

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()